# 02 — Движение плюма по картам TEC adjusted

Здесь вручную отмечаем **верхнюю границу одного и того же плюма** минимум в двух моментах UTC и измеряем движение между соседними моментами (08–09, 09–10).

Порядок работы:
1. Выберите время и посмотрите исходные карты TEC adjusted без линий разметки.
2. Задайте границы списками точек `(долгота, широта)` или мышью.
3. Посмотрите каждую границу на её карте и две границы вместе.
4. Проверьте центры, вспомогательные линии и направленную кратчайшую дугу между центрами.
5. Получите таблицу скоростей в **градусах в час**.

Автоматического детектора здесь пока нет. Контуры задаёт пользователь после просмотра данных.

## 1. Окружение

Выберите ядро Python с зависимостями проекта (`poetry install`). Тетрадь работает из корня проекта или из `notebooks/`.

Обычного inline-режима достаточно для ввода координат. Для ввода мышью нужен интерактивный backend: например, установленный `ipympl` и `%matplotlib widget` **до создания карт**, либо локальный `%matplotlib qt`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").exists() and (p / "app").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Откройте тетрадь из каталога Polar-lights или notebooks.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from io import BytesIO
from IPython.display import Image, display
from app.visualization.plume_tracking import (
    BoundaryPicker, boundary_center, boundary_points, estimate_motion,
    load_adjusted_tec, plot_boundary_pair, plot_motion_step, plot_plume_map, utc_time,
)

print("Корень проекта:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. Время и область карты

Используются локальные `tec_adjusted_*.h5` из `files/` и вложенных папок, в том числе файлы сразу за несколько суток. Загружаются только точные выбранные срезы. Если срез отсутствует, будет ошибка с ближайшими доступными моментами; незаметной подмены времени нет.

Если данных ещё нет, сначала загрузите продукт **Adjusted TEC** существующим загрузчиком проекта или через SIMuRG и поместите HDF5 в `files/`.

Все координаты — **географические**, в градусах. Для увеличения нужной области измените `MAP_EXTENT`. Для области через 180° используйте, например, `(160, 200, 30, 80)` и `CENTRAL_LONGITUDE = 180`.

In [ ]:
DATA_DIR = PROJECT_ROOT / "files"
TIMES = [
    "2026-01-20 10:00:00",
    "2026-01-20 11:00:00",
    "2026-01-20 12:00:00",
    "2026-01-20 13:00:00",
]
# MAP_EXTENT = (-30, 90, 30, 90)  # west, east, south, north; сузьте после обзора
MAP_EXTENT = (-20, 60, 40, 65)  # west, east, south, north; сузьте после обзора
CENTRAL_LONGITUDE = 0
COLOR_LIMITS = (0, 60)  # одна шкала TECU для всех моментов
POINT_SIZE = 10
OUTPUT_DIR = PROJECT_ROOT / "results" / "plume_tracking"
SAVE_RESULTS = False  # True: сохранить PNG, CSV и ручные границы JSON

TIMES = sorted(utc_time(t) for t in TIMES)
if len(TIMES) < 2 or len(set(TIMES)) != len(TIMES):
    raise ValueError("Задайте минимум два различных момента времени.")
MAP_OPTIONS = dict(
    extent=MAP_EXTENT, central_longitude=CENTRAL_LONGITUDE,
    color_limits=COLOR_LIMITS, point_size=POINT_SIZE,
)

In [ ]:
tec_data = load_adjusted_tec(DATA_DIR, TIMES)
display(pd.DataFrame([
    {"time_UTC": t, "TEC_points": len(tec_data[t])} for t in TIMES
]))

def show_map(time, line=None):
    """Достаточно времени; граница необязательна: show_map(TIMES[0], line)."""
    return plot_plume_map(tec_data, time, line, **MAP_OPTIONS)

def show_figure(fig, name):
    """Render before closing: Cartopy figures otherwise may keep only a colorbar."""
    png = BytesIO()
    fig.savefig(png, format="png", dpi=140)
    png_data = png.getvalue()
    if SAVE_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / f"{name}.png").write_bytes(png_data)
    display(Image(data=png_data))
    plt.close(fig)

## 3. Исходные карты без разметки

На этих картах присутствует исходный TEC adjusted, но ещё нет введённых границ, центров и треугольников. Сначала найдите плюм и при необходимости сузьте `MAP_EXTENT`, затем повторите ячейки параметров, загрузки и карт.

In [ ]:
for time in TIMES:
    fig, ax = show_map(time)
    show_figure(fig, f"01_tec_{time:%Y%m%d_%H%M%S}")

## 4. Ручная верхняя граница

Вместо `None` впишите минимум две точки **вдоль верхней границы**, в порядке обхода. Первая координата — долгота, вторая — широта. Границу не нужно замыкать. Её центр обозначается крестом: это точка на половине длины ломаной, а не среднее координат вершин.

Пример **формата**, не разметка реальных данных: `[(-110, 50), (-105, 53), (-100, 54)]`.

Добавляя моменты в `TIMES`, добавляйте и соответствующие строки ниже. `None` разрешён для просмотра карты; скорость для пары без обеих границ не вычисляется. Проверяйте, что на всех картах отмечен один и тот же плюм.

In [ ]:
# Заполните после просмотра исходных карт. Можно использовать UTC datetime или полную строку времени.
LINES = {
    TIMES[0]: [(18, 62), (58, 55)],
    TIMES[1]: [(3, 62), (55, 52)],
    TIMES[2]: [(-10, 62), (50, 50)],
    TIMES[3]: [(-18, 61), (45, 45)],
} if len(TIMES) >= 3 else {time: None for time in TIMES}

# Например: LINES[TIMES[0]] = [(-110, 50), (-105, 53), (-100, 54)]
normalized_lines = {utc_time(time): line for time, line in LINES.items()}
unknown_times = set(normalized_lines) - set(TIMES)
if unknown_times:
    raise ValueError(f"В LINES есть моменты, не включённые в TIMES: {unknown_times}")
observations = [{"time": time, "line": normalized_lines.get(time)} for time in TIMES]
for item in observations:
    if item["line"] is not None:
        boundary_points(item["line"])  # сразу проверяем форму, диапазоны и ненулевую длину

### Необязательно: ввод мышью

Если удобнее рисовать, включите интерактивный backend, задайте `USE_MOUSE = True` и выберите индекс момента. Выполните следующую ячейку отдельно: **левая кнопка** добавляет точку, **правая** удаляет последнюю, **Enter** завершает линию. Панорамирование и масштабирование инструментами Matplotlib выключите на время разметки.

После разметки выполните ячейку переноса точек, затем повторите для остальных моментов. Полученный список печатается: его можно перенести в `LINES`, чтобы разметка сохранилась в исходном коде тетради. При повторном запуске ячейки `LINES` несохранённая разметка мышью заменится её значениями.

In [ ]:
USE_MOUSE = False
PICK_INDEX = 0
picker = None
if USE_MOUSE:
    backend = matplotlib.get_backend().lower()
    if "inline" in backend or backend == "agg":
        raise RuntimeError("Для мыши включите %matplotlib widget (нужен ipympl) или %matplotlib qt.")
    fig, ax = show_map(observations[PICK_INDEX]["time"])
    picker = BoundaryPicker(ax, central_longitude=CENTRAL_LONGITUDE)
    plt.show()

In [ ]:
# Выполните после кликов на карте. При USE_MOUSE=False ячейка ничего не меняет.
if USE_MOUSE and picker is not None:
    observations[PICK_INDEX]["line"] = picker.finish()
    LINES[observations[PICK_INDEX]["time"]] = observations[PICK_INDEX]["line"]
    print("Время:", observations[PICK_INDEX]["time"])
    print("Линия:", observations[PICK_INDEX]["line"])

## 5. Каждая граница на своём TEC adjusted

Все карты используют одну область и цветовую шкалу. Крест — центр границы. Отсутствующая линия не мешает просмотреть исходную карту.

In [ ]:
for item in observations:
    fig, ax = show_map(item["time"], item["line"])
    show_figure(fig, f"02_boundary_{item['time']:%Y%m%d_%H%M%S}")

## 6. Две соседние границы одновременно на одной карте

Пурпурная линия — первый момент, голубая — второй. Фон всегда **TEC adjusted второго момента**, время указано в заголовке. Это наложение контуров, значения TEC двух моментов не смешиваются.

Если между размеченными моментами есть момент без линии, он не пропускается ради соединения несоседних наблюдений.

In [ ]:
for first, second in zip(observations, observations[1:]):
    if first["line"] is None or second["line"] is None:
        print(f"{first['time']:%H:%M} → {second['time']:%H:%M}: задайте обе границы.")
        continue
    fig, ax = plot_boundary_pair(
        tec_data, first["time"], first["line"], second["time"], second["line"], **MAP_OPTIONS,
    )
    show_figure(fig, f"03_overlay_{first['time']:%Y%m%d_%H%M%S}_{second['time']:%Y%m%d_%H%M%S}")

## 7. Геометрия и расчёт

Обозначения: $C_1=(\lambda_1,\varphi_1)$ и $C_2=(\lambda_2,\varphi_2)$ — центры границ.

1. Из $C_1$ проводим вертикаль $\lambda=\lambda_1$ до **точки $Q$ на второй границе**. Если пересечений несколько, выбираем ближайшее к $C_1$; при равенстве расстояний — с меньшей широтой. Пересечение ищется на сегментах ломаной, а не только среди введённых вершин.
2. Из $C_2$ проводим горизонталь к этой вертикали: $P=(\lambda_1,\varphi_2)$. Угол в $P$ равен 90°. **$Q$ и $P$ могут различаться.** При необходимости вертикаль продолжается до $P$; $P$ не обязан лежать на границе.
3. $C_1P$ и $PC_2$ показывают изменения широты и долготы. Скорость вычисляется по **кратчайшей дуге большого круга** между $C_1$ и $C_2$. На карте PlateCarree эта дуга может быть изогнутой.

Земля моделируется сферой. Пусть $\Delta\lambda=\lambda_2-\lambda_1$ — кратчайшая разность долгот, $\Delta\varphi=\varphi_2-\varphi_1$. Центральный угол между центрами вычисляется по сферической геометрии (углы внутри тригонометрических функций — в радианах):

$$a=\sin^2\frac{\Delta\varphi}{2}+\cos\varphi_1\cos\varphi_2\sin^2\frac{\Delta\lambda}{2},\qquad \sigma=2\operatorname{atan2}(\sqrt a,\sqrt{1-a}).$$

$$d_{\mathrm{deg}}=\sigma\frac{180}{\pi},\qquad v=\frac{d_{\mathrm{deg}}}{(t_2-t_1)/1\text{ час}}.$$

Направление — **начальный азимут кратчайшей дуги в $C_1$**: по часовой стрелке от географического севера (0° — север, 90° — восток, 180° — юг, 270° — запад). Азимут вдоль дуги может меняться; угол стрелки на плоской карте не следует измерять транспортиром для получения азимута.

**Физический смысл:** `speed_deg_h` — среднее угловое смещение выбранного центра за час относительно центра Земли. Схождение меридианов учитывается сферической формулой автоматически; дополнительно умножать результат на косинус не нужно. Например, между точками (0°, 60°) и (10°, 60°) кратчайшая дуга составляет около 4.995°, поэтому за час скорость равна около 4.995°/ч. Для малых смещений это согласуется с приближением $\sqrt{(\Delta\varphi)^2+(\cos\bar\varphi\,\Delta\lambda)^2}$. Радиус и высота слоя для °/ч не нужны; для перевода в км/ч потребовался бы радиус выбранного слоя.

**Роль треугольника:** линии по меридиану и параллели помогают видеть изменения координат, но скорость не вычисляется по их плоской гипотенузе. Теорема Пифагора в координатах долгота/широта здесь не применяется.

Центр по-прежнему выбирается на половине длины нарисованной ломаной в координатах долгота/широта. Это явное правило выбора опорной точки; сферическая метрика применяется к перемещению между такими точками. Изменение формы или длины ручного контура может смещать центр. Результат характеризует наблюдаемое перемещение выбранной особенности плюма между двумя моментами, а не измеренную скорость отдельных частиц плазмы или полную длину неизвестной траектории.

Если вертикаль не пересекает вторую границу, точка $Q$ отсутствует, но сферическая скорость между центрами вычисляется. При совпадении центров скорость нулевая, направление не определено. Для диаметрально противоположных точек расстояние равно 180°, единственной кратчайшей дуги нет. В начальном географическом полюсе азимут также не определён; такие случаи отмечаются в таблице.

О сферических геодезических и начальном азимуте: [документация PROJ](https://proj.org/en/stable/geodesic.html).

In [ ]:
motions = []
diagnostics = []
for first, second in zip(observations, observations[1:]):
    reason = None
    if first["line"] is None or second["line"] is None:
        reason = "Не заданы обе границы."
    else:
        try:
            motions.append(estimate_motion(
                first["time"], first["line"], second["time"], second["line"],
            ))
        except ValueError as error:
            reason = str(error)
    if reason:
        diagnostics.append({"t1_UTC": first["time"], "t2_UTC": second["time"], "причина": reason})

if diagnostics:
    display(pd.DataFrame(diagnostics))
if not motions:
    print("Нет рассчитанных пар. Заполните границы и повторите шаги 5–9.")

## 8. Каждый шаг построения

Для каждой пары выводятся четыре карты: **центры → вертикаль до $Q$ (если пересечение есть) → вспомогательные линии сетки → направленная кратчайшая дуга**. Жёлтая точка $Q$ находится на второй границе, чёрная точка $P$ — вершина прямого угла вспомогательного построения. Пунктир показывает изменения координат. Сплошная дуга со стрелкой идёт от $C_1$ к $C_2$ и соответствует расстоянию, использованному для скорости. Тёмные линии имеют белую обводку для контраста на фоне TEC.

In [ ]:
for motion in motions:
    fig, axes = plt.subplots(
        2, 2, figsize=(24, 12),
        subplot_kw={"projection": ccrs.PlateCarree(central_longitude=CENTRAL_LONGITUDE)},
    )
    for step, ax in enumerate(axes.flat):
        plot_motion_step(tec_data, motion, step=step, ax=ax, **MAP_OPTIONS)
    fig.suptitle(
        f"{motion.time1:%Y-%m-%d %H:%M:%S} → {motion.time2:%Y-%m-%d %H:%M:%S} UTC",
        fontsize=15,
    )
    fig.subplots_adjust(left=0.04, right=0.90, bottom=0.06, top=0.90, wspace=0.14, hspace=0.28)
    colorbar_axis = fig.add_axes([0.93, 0.12, 0.012, 0.72])
    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=plt.Normalize(*COLOR_LIMITS), cmap="jet"),
        cax=colorbar_axis,
    )
    colorbar.set_label("TEC adjusted, TECU")
    show_figure(fig, f"04_steps_{motion.time1:%Y%m%d_%H%M%S}_{motion.time2:%Y%m%d_%H%M%S}")

## 9. Скорость и направление по соседним моментам

Таблица содержит время начала и конца интервала, среднюю сферическую скорость `speed_deg_h` в °/ч и краткое направление `direction` (С, СВ, В, …). При `SAVE_RESULTS=True` сохраняются эта же таблица CSV, границы JSON и показанные карты PNG.

In [ ]:
TABLE_COLUMNS = ["t1_UTC", "t2_UTC", "speed_deg_h", "direction"]
velocity_table = pd.DataFrame([motion.as_record() for motion in motions]).reindex(columns=TABLE_COLUMNS)
if not velocity_table.empty:
    display(velocity_table.round(4))
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if not velocity_table.empty:
        velocity_table.to_csv(OUTPUT_DIR / "plume_velocity.csv", index=False)
    if diagnostics:
        pd.DataFrame(diagnostics).to_csv(OUTPUT_DIR / "skipped_pairs.csv", index=False)
    annotations = {
        "coordinate_system": "geographic_lon_lat_degrees",
        "metric": "spherical_great_circle",
        "direction_method": "initial_great_circle_bearing",
        "center_method": "half_polyline_length_lon_lat_degrees",
        "observations": [
            {"time": item["time"].isoformat(),
             "line": None if item["line"] is None else np.asarray(item["line"]).tolist()}
            for item in observations
        ],
    }
    (OUTPUT_DIR / "boundaries.json").write_text(
        json.dumps(annotations, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    print("Результаты:", OUTPUT_DIR)